In [ ]:
# Import dependencies
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
plt.style.use('ggplot')

In [ ]:
# Get the data file names
os.listdir('data')

In [ ]:
# Import data
jan = pd.read_csv('data/201901-citibike-tripdata.csv',)
feb = pd.read_csv('data/201902-citibike-tripdata.csv',)
jun = pd.read_csv('data/201906-citibike-tripdata.csv',)
jul = pd.read_csv('data/201907-citibike-tripdata.csv',)
aug = pd.read_csv('data/201908-citibike-tripdata.csv',)
dec = pd.read_csv('data/201912-citibike-tripdata.csv')

display(jan.head())
display(feb.head())
display(jun.head())
display(jul.head())
display(aug.head())
dec.head()

In [ ]:
# Check if all of the column titles in all of the DataFrames are exactly the same both in spelling and in order
display((list(jan.columns) == list(feb.columns)) & (list(jan.columns) == list(jun.columns)) & (list(jan.columns) == list(jul.columns)) \
& (list(jan.columns) == list(aug.columns)) & (list(jan.columns) == list(dec.columns)))

# Check if all of the columns have the same data types in all of the DataFrames
(jan.dtypes == feb.dtypes) & (jan.dtypes == jun.dtypes) & (jan.dtypes == jul.dtypes) & (jan.dtypes == aug.dtypes) & (jan.dtypes == dec.dtypes)

In [ ]:
# Find out which DataFrames have different data types for 'start station id' and 'end station id' 
print(f"January data types:\n{jan[['start station id', 'end station id']].dtypes}\n")
print(f"February data types:\n{feb[['start station id', 'end station id']].dtypes}\n")
print(f"June data types:\n{jun[['start station id', 'end station id']].dtypes}\n")
print(f"July data types:\n{jul[['start station id', 'end station id']].dtypes}\n")
print(f"August data types:\n{aug[['start station id', 'end station id']].dtypes}\n")
print(f"December data types:\n{dec[['start station id', 'end station id']].dtypes}")

In [ ]:
# Check if any decimal values in 'start station id' and 'end station id' are not '0' or nan
print(jan['start station id'].astype(str).str.split('.').str[1].unique())
print(feb['start station id'].astype(str).str.split('.').str[1].unique())
print(jul['start station id'].astype(str).str.split('.').str[1].unique())
print(aug['start station id'].astype(str).str.split('.').str[1].unique())

print(jan['end station id'].astype(str).str.split('.').str[1].unique())
print(feb['end station id'].astype(str).str.split('.').str[1].unique())
print(jul['end station id'].astype(str).str.split('.').str[1].unique())
print(aug['end station id'].astype(str).str.split('.').str[1].unique())

In [ ]:
# Missing values must be removed before data types can be changed
# Check if jun and dec DataFrames have missing values
print(f"jun has NaNs: {jun.isnull().values.any()}")
print(f"dec has NaNs: {dec.isnull().values.any()}") 

# Check which rows have missing values
print(f"\njan counts pre-.dropna():\n{jan.count()}\n")
print(f"feb counts pre-.dropna():\n{feb.count()}\n")
print(f"jul counts pre-.dropna():\n{jul.count()}\n")
print(f"aug counts pre-.dropna():\n{aug.count()}")

# Remove missing values
jan.dropna(inplace=True)
feb.dropna(inplace=True)
jul.dropna(inplace=True)
aug.dropna(inplace=True)

In [ ]:
# Convert the 'start station id' and 'end station id' columns' data types to integer
jan = jan.astype({'start station id':'int64', 'end station id':'int64'})
feb = feb.astype({'start station id':'int64', 'end station id':'int64'})
jul = jul.astype({'start station id':'int64', 'end station id':'int64'})
aug = aug.astype({'start station id':'int64', 'end station id':'int64'})

# Confirm the data types were changed
print(f"jan data types:\n{jan[['start station id', 'end station id']].dtypes}\n")
print(f"feb data types:\n{feb[['start station id', 'end station id']].dtypes}\n")
print(f"jul data types:\n{jul[['start station id', 'end station id']].dtypes}\n")
print(f"aug data types:\n{aug[['start station id', 'end station id']].dtypes}")

In [ ]:
# Combine data
all_data = pd.concat([jan, feb, jun, jul, aug, dec], ignore_index=True)

# Use a sample of 10% of the entire dataset for smoother analysis
sample_data = all_data.sample(frac=0.1)

display(all_data.head())
sample_data.head()

In [ ]:
# Generate descriptive statistics
# The min 'birth year' value doesn't make sense; the oldest rider is 134 years old
all_data.describe()

In [ ]:
# Separate column name words with '_' to improve comprehensibility
sample_data.rename(columns={
    'tripduration':'trip_duration',
    'starttime':'start_time',
    'stoptime':'stop_time',
    'bikeid':'bike_id',
    'usertype':'user_type'
}, inplace=True)

# Replace white spaces in the column names with '_' for consistency
sample_data.columns = sample_data.columns.str.replace(' ', '_')

# Confirm the columns were renamed correctly
sample_data.columns

In [ ]:
# Check that all of the columns have the proper data types
display(sample_data.dtypes)

# Convert the 'start_time' and 'stop_time' columns to the datetime data type
sample_data['start_time'] = pd.to_datetime(sample_data['start_time'])
sample_data['stop_time'] = pd.to_datetime(sample_data['stop_time'])

# Confirm the data types were changed
sample_data.dtypes

In [ ]:
# Replace the numeric values in the 'gender' column with the name of each gender to improve comprehensibility
sample_data['gender'].replace(0, 'Unknown', inplace=True)
sample_data['gender'].replace(1, 'Male', inplace=True)
sample_data['gender'].replace(2, 'Female', inplace=True)

# Confirm that the 'gender' column's values were replaced
print(f"gender values: {sample_data['gender'].unique()}")

# Confirm that the 'user_type' column only includes "Subscriber" and "Customer" values
print(f"user_type values: {sample_data['user_type'].unique()}\n")

# Determine what are the correct 'birth_year' values to keep
year_ticks = np.arange(1919, 2006, 5)
sample_data['birth_year'].plot(kind='box', figsize=(10,10), grid=True, yticks=year_ticks, showmeans=True)
plt.show()

In [ ]:
# Get the outlier values
# Calculate the interquartile range
birth_year_sample = sample_data['birth_year']
q1_sample = np.percentile(birth_year_sample, 25)
q3_sample = np.percentile(birth_year_sample, 75)
iqr_sample = q3_sample - q1_sample

# Calculate the outliers
lower_boundary_sample = q1_sample - (1.5 * iqr_sample)
lower_boundary_outliers_sample = birth_year_sample[birth_year_sample <= lower_boundary_sample].unique()
lower_boundary_outliers_sample.sort()

# View the outliers
max_outlier_sample = lower_boundary_outliers_sample.max()
display(lower_boundary_outliers_sample)
print(f"\nMax outlier value: {max_outlier_sample}")

# Remove all of the 'birth_year' values before and including the highest outlier value
sample_data = sample_data.loc[sample_data['birth_year'] > max_outlier_sample, :]
print(f"New oldest age = {2019 - sample_data['birth_year'].min()}")

# Apply cleansing to all_data

In [ ]:
# Separate column name words with '_' to improve comprehensibility
all_data.rename(columns={
    'tripduration':'trip_duration',
    'starttime':'start_time',
    'stoptime':'stop_time',
    'bikeid':'bike_id',
    'usertype':'user_type'
}, inplace=True)

# Replace white spaces in the column names with '_' for consistency
all_data.columns = all_data.columns.str.replace(' ', '_')

# Confirm the columns were renamed correctly
all_data.columns

In [ ]:
# Convert the 'start_time' and 'stop_time' columns to the datetime data type
all_data['start_time'] = pd.to_datetime(all_data['start_time'])
all_data['stop_time'] = pd.to_datetime(all_data['stop_time'])

# Confirm the data types were changed
display(all_data.dtypes)

# Replace the numeric values in the 'gender' column with the name of each gender to improve comprehensibility
all_data['gender'].replace(0, 'Unknown', inplace=True)
all_data['gender'].replace(1, 'Male', inplace=True)
all_data['gender'].replace(2, 'Female', inplace=True)

# Confirm that the 'gender' column's values were replaced
print(f"gender values: {all_data['gender'].unique()}")

# Confirm that the 'user_type' column only includes "Subscriber" and "Customer" values
print(f"user_type values: {all_data['user_type'].unique()}\n")

# Determine what are the correct 'birth_year' values to keep
all_data['birth_year'].plot(kind='box', figsize=(10,10), grid=True, yticks=year_ticks, showmeans=True)
plt.show()

In [ ]:
# Get the outlier values
# Calculate the interquartile range
birth_year = all_data['birth_year']
q1 = np.percentile(birth_year, 25)
q3 = np.percentile(birth_year, 75)
iqr = q3_sample - q1_sample

# Calculate the outliers
lower_boundary = q1 - (1.5 * iqr)
lower_boundary_outliers = birth_year[birth_year <= lower_boundary].unique()
lower_boundary_outliers.sort()

# View the outliers
max_outlier = lower_boundary_outliers.max()
display(lower_boundary_outliers)
print(f"\nMax outlier value: {max_outlier}")

# Remove all of the 'birth_year' values before and including the highest outlier value
all_data = all_data.loc[all_data['birth_year'] > max_outlier, :]
print(f"New oldest age = {2019 - all_data['birth_year'].min()}")

In [ ]:
# Export sample_data DataFrame to CSV file 
sample_data.to_csv('data/sample_data.csv', index=False)

In [ ]:
# Export all_data DataFrame to CSV file
all_data.to_csv('data/all_data.csv', index=False)